# Evaluation Report — Clinical QA System
**LJMU MSc ML\&AI | Amit Choudhary (1187247)**

Full evaluation of all three experimental conditions with thesis-quality charts.
Results address the primary research question:
> *Can a RAG + hallucination-filter system reduce hallucination by ≥50% vs. a base LLM
> while maintaining ≥75% MedQA accuracy and sub-3s P95 latency?*

In [ ]:
import sys, json, math
from pathlib import Path
sys.path.insert(0, "..")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from src.config import settings
from src.evaluation.hallucination_eval import load_predictions, compute_hallucination_stats

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.dpi"] = 130
plt.rcParams["font.size"] = 11

CONDITIONS = ["base_llm", "rag_only", "rag_with_hallucination_filter"]
LABELS = {
    "base_llm": "Base LLM",
    "rag_only": "RAG Only",
    "rag_with_hallucination_filter": "RAG + Halluc Filter
(Proposed)",
}
COLORS = ["#d73027", "#fee090", "#1a9850"]
print("Setup complete.")

## 1. Load All Results

In [ ]:
data = {}
for c in CONDITIONS:
    preds = load_predictions(c)
    if preds:
        stats = compute_hallucination_stats(preds)
        accs  = [p.get("correct", False) for p in preds]
        times = [p.get("response_time_s", 0) for p in preds]
        data[c] = {
            "preds": preds,
            "accuracy": np.mean(accs) if accs else None,
            "halluc_mean": stats.get("mean"),
            "halluc_std":  stats.get("std"),
            "halluc_p95":  stats.get("p95"),
            "avg_time":    np.mean(times) if times else None,
            "p95_time":    np.percentile(times, 95) if times else None,
            "n":           len(preds),
        }
        print(f"{c}: n={len(preds)}, acc={data[c]['accuracy']:.4f}, halluc={data[c]['halluc_mean']:.4f}")
    else:
        print(f"{c}: no data — run scripts/run_full_benchmark.py --n 500")

## 2. Table 4.1 — Main Results

In [ ]:
rows = []
for c in CONDITIONS:
    d = data.get(c, {})
    rows.append({
        "Condition":       LABELS.get(c, c).replace("\n", " "),
        "N":               d.get("n"),
        "Accuracy":        round(d["accuracy"],    4) if d.get("accuracy")    is not None else None,
        "Halluc Rate":     round(d["halluc_mean"],  4) if d.get("halluc_mean") is not None else None,
        "Halluc Std":      round(d["halluc_std"],   4) if d.get("halluc_std")  is not None else None,
        "Avg Time (s)":    round(d["avg_time"],     2) if d.get("avg_time")    is not None else None,
        "P95 Time (s)":    round(d["p95_time"],     2) if d.get("p95_time")    is not None else None,
    })
df = pd.DataFrame(rows)
print("
Table 4.1 — Experimental Conditions Comparison")
display(df)
df.to_csv("../docs/table_4_1_comparison.csv", index=False)

## 3. Hypothesis Tests

In [ ]:
# Primary hypothesis: hallucination reduction >= 50%
base_d = data.get("base_llm", {})
prop_d = data.get("rag_with_hallucination_filter", {})
base_h = base_d.get("halluc_mean")
prop_h = prop_d.get("halluc_mean")

print("H1: Hallucination Reduction >= 50%")
if base_h is not None and prop_h is not None:
    reduction = (base_h - prop_h) / base_h * 100 if base_h > 0 else 0
    print(f"  Base LLM hallucination rate:     {base_h:.4f}")
    print(f"  Proposed system hallucination:   {prop_h:.4f}")
    print(f"  Reduction:                       {reduction:.1f}%")
    print(f"  RESULT: {chr(10003) if reduction >= 50 else chr(215)} ('{'SUPPORTED' if reduction >= 50 else 'NOT SUPPORTED'}' at 50% threshold)")
else:
    print("  — data not yet available —")

print()
print("H2: Accuracy >= 75%")
prop_acc = prop_d.get("accuracy")
if prop_acc is not None:
    print(f"  Proposed system accuracy: {prop_acc:.4f}")
    print(f"  RESULT: {chr(10003) if prop_acc >= 0.75 else chr(215)} ('{'SUPPORTED' if prop_acc >= 0.75 else 'NOT SUPPORTED'}' at 75% threshold)")
else:
    print("  — data not yet available —")

print()
print("H3: P95 Response Time < 3s")
prop_p95 = prop_d.get("p95_time")
if prop_p95 is not None:
    print(f"  Proposed system P95 time: {prop_p95:.2f}s")
    print(f"  RESULT: {chr(10003) if prop_p95 < 3.0 else chr(215)} ('{'SUPPORTED' if prop_p95 < 3.0 else 'NOT SUPPORTED'}' at 3s SLA)")
else:
    print("  — data not yet available —")

## 4. Main Results Figure (Thesis Figure 4.1)

In [ ]:
avail = {c: d for c, d in data.items() if d.get("accuracy") is not None}

if len(avail) >= 2:
    lbls   = [LABELS[c] for c in avail]
    accs   = [avail[c]["accuracy"]   for c in avail]
    halluc = [avail[c]["halluc_mean"] for c in avail]
    h_std  = [avail[c]["halluc_std"]  for c in avail]
    times  = [avail[c]["avg_time"]    for c in avail]
    p95s   = [avail[c]["p95_time"]    for c in avail]
    cols   = COLORS[:len(avail)]

    fig = plt.figure(figsize=(16, 10))
    gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.42, wspace=0.36)

    # (a) Accuracy
    ax0 = fig.add_subplot(gs[0, 0])
    ax0.bar(range(len(lbls)), accs, color=cols, alpha=0.85, edgecolor="black")
    ax0.axhline(0.75, color="red", linestyle="--", lw=1.5, label="Target 75%")
    ax0.set_xticks(range(len(lbls))); ax0.set_xticklabels(lbls, fontsize=9)
    ax0.set_ylabel("MedQA Accuracy"); ax0.set_title("(a) MedQA Accuracy", fontweight="bold")
    ax0.set_ylim(0, 1.0); ax0.legend(fontsize=9)
    for i, a in enumerate(accs): ax0.text(i, a+0.01, f"{a:.3f}", ha="center", fontsize=9, fontweight="bold")

    # (b) Hallucination rate with std bars
    ax1 = fig.add_subplot(gs[0, 1])
    ax1.bar(range(len(lbls)), halluc, yerr=h_std, capsize=5, color=cols, alpha=0.85, edgecolor="black")
    ax1.set_xticks(range(len(lbls))); ax1.set_xticklabels(lbls, fontsize=9)
    ax1.set_ylabel("Hallucination Rate"); ax1.set_title("(b) Hallucination Rate", fontweight="bold")
    for i, h in enumerate(halluc): ax1.text(i, h+max(h_std)*0.12+0.003, f"{h:.3f}", ha="center", fontsize=9, fontweight="bold")

    # (c) Response time
    ax2 = fig.add_subplot(gs[1, 0])
    x = np.arange(len(lbls)); w = 0.35
    ax2.bar(x-w/2, times, w, label="Mean", color="steelblue", alpha=0.8)
    ax2.bar(x+w/2, p95s,  w, label="P95",  color="coral",     alpha=0.8)
    ax2.axhline(3.0, color="red", linestyle="--", lw=1.5, label="SLA 3s")
    ax2.set_xticks(x); ax2.set_xticklabels(lbls, fontsize=9)
    ax2.set_ylabel("Time (s)"); ax2.set_title("(c) Response Time", fontweight="bold")
    ax2.legend(fontsize=9)

    # (d) Hallucination distribution
    ax3 = fig.add_subplot(gs[1, 1])
    for i, c in enumerate(avail):
        rates = [p.get("hallucination_rate", 0) for p in avail[c]["preds"]]
        ax3.hist(rates, bins=25, alpha=0.55, label=LABELS[c].replace("\n", " "), color=COLORS[i], density=True)
    ax3.set_xlabel("Hallucination Rate"); ax3.set_ylabel("Density")
    ax3.set_title("(d) Hallucination Distribution", fontweight="bold")
    ax3.legend(fontsize=8)

    plt.suptitle("Figure 4.1 — Clinical QA System: Full Benchmark Results", fontsize=13, fontweight="bold")
    Path("../docs").mkdir(exist_ok=True)
    plt.savefig("../docs/figure_4_1_benchmark_results.png", bbox_inches="tight", dpi=150)
    plt.show()
    print("Saved → docs/figure_4_1_benchmark_results.png")
else:
    print("Run: python scripts/run_full_benchmark.py --n 500")

## 5. RAGAS Results

In [ ]:
# Load RAGAS metrics if available
ragas_path = settings.data_processed_dir / "ragas_results.json"
if ragas_path.exists():
    rm = json.loads(ragas_path.read_text())
    ragas_labels = ["Faithfulness", "Answer
Relevancy", "Context
Precision", "Context
Recall"]
    ragas_vals   = [rm.get("faithfulness",0), rm.get("answer_relevancy",0),
                    rm.get("context_precision",0), rm.get("context_recall",0)]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(ragas_labels, ragas_vals, color="#1a9850", alpha=0.85, edgecolor="black")
    ax.set_ylim(0, 1.0)
    ax.set_title("Figure 4.2 — RAGAS Metrics (Proposed System)", fontweight="bold")
    ax.set_ylabel("Score")
    for i, v in enumerate(ragas_vals):
        ax.text(i, v+0.01, f"{v:.3f}", ha="center", fontsize=10, fontweight="bold")
    plt.savefig("../docs/figure_4_2_ragas.png", bbox_inches="tight", dpi=150)
    plt.show()
    print(f"Faithfulness: {rm.get('faithfulness', 0):.4f}")
    print(f"Answer Relevancy: {rm.get('answer_relevancy', 0):.4f}")
    print(f"Context Precision: {rm.get('context_precision', 0):.4f}")
    print(f"Context Recall: {rm.get('context_recall', 0):.4f}")
else:
    print("RAGAS results not yet available. Run: python scripts/run_full_benchmark.py --phase ragas")

## 6. Accuracy vs Hallucination Rate Trade-off

In [ ]:
avail2 = {c: d for c, d in data.items() if d.get("accuracy") is not None and d.get("halluc_mean") is not None}
if avail2:
    fig, ax = plt.subplots(figsize=(8, 6))
    for i, (c, d) in enumerate(avail2.items()):
        ax.scatter(d["halluc_mean"], d["accuracy"],
                   s=200, color=COLORS[i], edgecolors="black", linewidths=1.5, zorder=5)
        ax.annotate(LABELS[c].replace("\n", " "),
                    (d["halluc_mean"], d["accuracy"]),
                    textcoords="offset points", xytext=(10, 5), fontsize=10)
    ax.axhline(0.75, color="blue", linestyle="--", alpha=0.5, label="Accuracy target 75%")
    ax.set_xlabel("Hallucination Rate", fontsize=12)
    ax.set_ylabel("MedQA Accuracy", fontsize=12)
    ax.set_title("Figure 4.3 — Accuracy vs Hallucination Trade-off", fontweight="bold", fontsize=13)
    ax.legend(fontsize=10)
    plt.savefig("../docs/figure_4_3_tradeoff.png", bbox_inches="tight", dpi=150)
    plt.show()
else:
    print("No data available yet.")

## 7. Summary for Thesis Chapter 4

In [ ]:
print("="*65)
print("  Chapter 4 Summary — Key Findings")
print("="*65)

base_d = data.get("base_llm", {})
prop_d = data.get("rag_with_hallucination_filter", {})

if base_d.get("accuracy") and prop_d.get("accuracy"):
    acc_gain   = prop_d["accuracy"] - base_d["accuracy"]
    h_red      = (base_d["halluc_mean"] - prop_d["halluc_mean"]) / base_d["halluc_mean"] * 100
    sla_ok     = prop_d.get("p95_time", 999) < 3.0
    acc_target = prop_d["accuracy"] >= 0.75
    h_target   = h_red >= 50.0

    print(f"  Accuracy (Base → Proposed):  {base_d['accuracy']:.4f} → {prop_d['accuracy']:.4f} ({acc_gain:+.4f})")
    print(f"  Hallucination reduction:     {h_red:.1f}% ",
          chr(10003) if h_target else chr(215))
    print(f"  Accuracy target (>=75%):     ", chr(10003) if acc_target else chr(215))
    print(f"  SLA (P95 < 3s):              ", chr(10003) if sla_ok else chr(215))
    print()
    print("  All three research hypotheses:")
    for label, met in [("H1: Hallucination -50%", h_target),
                        ("H2: Accuracy >= 75%", acc_target),
                        ("H3: P95 < 3s", sla_ok)]:
        print(f"    {label}: {'SUPPORTED' if met else 'NOT SUPPORTED'}")
else:
    print("  Run the full benchmark first:")
    print("    python scripts/run_full_benchmark.py --n 500")
print("="*65)